In [30]:
## This notebook shows how to construct an explained variance plot using a PMD decomposition
import masknmf
import torch
import os
import numpy as np
import sys
import matplotlib.pyplot as plt


##In the existing folder
import iblwfci_vis
import iblwfci_utils
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Define some basic utility functions for working with factorizations (Joao's factorizations or the masknmf ideas)

In [11]:
def absorb_pixel_normalizer(U_sparse, pixel_normalizer):
    """
    U_sparse: (N_pixels, R) sparse COO
    pixel_normalizer: (N_pixels,) dense tensor

    Returns:
        U_scaled: sparse COO
    """
    U_sparse = U_sparse.coalesce()
    indices = U_sparse.indices()
    values = U_sparse.values()

    row_indices = indices[0]
    scaled_values = values * pixel_normalizer[row_indices]

    return torch.sparse_coo_tensor(
        indices,
        scaled_values,
        U_sparse.shape,
        device=U_sparse.device
    )

def absorb_mean(U_scaled_sparse, V_dense, pixelwise_mean):
    """
    U_scaled_sparse: (N, R) sparse
    V_dense: (R, T) dense
    pixelwise_mean: (N,) dense

    Returns:
        U_new (sparse), V_new (dense)
    """

    N, R = U_scaled_sparse.shape
    T = V_dense.shape[1]

    # --- append mean as new sparse column ---
    U_scaled_sparse = U_scaled_sparse.coalesce()
    indices = U_scaled_sparse.indices()
    values = U_scaled_sparse.values()

    # New column index = R
    mean_col_indices = torch.stack([
        torch.arange(N, device=pixelwise_mean.device),
        torch.full((N,), R, device=pixelwise_mean.device)
    ])

    mean_values = pixelwise_mean

    new_indices = torch.cat([indices, mean_col_indices], dim=1)
    new_values = torch.cat([values, mean_values])

    U_new = torch.sparse_coo_tensor(
        new_indices,
        new_values,
        (N, R + 1),
        device=U_scaled_sparse.device
    )

    # --- append ones row to V ---
    ones_row = torch.ones(1, T, device=V_dense.device)
    V_new = torch.cat([V_dense, ones_row], dim=0)

    return U_new.coalesce(), V_new

def compute_mean(u, v):
    return torch.sparse.mm(u, torch.mean(v, dim = 1, keepdims = True)).flatten()

In [22]:
def get_pca_spectrum_pmd_obj(pmd_obj, device = 'cpu'):
    pixelwise_normalizer = pmd_obj.var_img.to(device).flatten()
    u = pmd_obj.u.to(device)
    v = pmd_obj.v.to(device)

    print('1')
    u_absorbed  = absorb_pixel_normalizer(u, pixelwise_normalizer)
    print('2')
    u_absorbed = u_absorbed.to(device)
    print('3')
    new_mean = compute_mean(u_absorbed, v)
    print('4')
    u_final, v_final = absorb_mean(u_absorbed, v, new_mean * -1)

    u_final = u_final.to(device)
    v_final = v_final.to(device)

    print('factorization started')
    r, s, v = masknmf.compression.decomposition.compute_lowrank_factorized_svd(u_final, v_final)
    print('completed')

    s = s.cpu().numpy()
        
    #Returns the explained variance spectrum
    return np.cumsum(s**2) / np.sum(s**2)
    

In [17]:
hemocorr_pmd = masknmf.PMDArray.from_hdf5("felicia_jan_26_008/hemocorr.hdf5")
hemocorr_pmd.to('cuda')
hemocorr_pmd.rescale = True

In [28]:
parent_path = "/data/lab/IBL_Alyx/churchlandlab_ucla/Subjects/FD_24/2023-06-08/001/raw_widefield_data"
u_path = os.path.join(parent_path, "U.npy")
svt_path = os.path.join(parent_path, "SVT.npy")
svtcorr_path = os.path.join(parent_path, "SVTcorr.npy")
frames_avg_path = os.path.join(parent_path, "frames_average.npy")

manual_mask = np.load(os.path.join(parent_path, "manual_mask.npy"))

In [31]:
joao_gcamp, joao_blood, joao_hemocorr = iblwfci_utils.load_joao_results(u_path,
                  svt_path,
                  svtcorr_path,
                  frames_avg_path,
                  functional_channel = 0)

In [23]:
explained_var_spectrum_amol = get_pca_spectrum_pmd_obj(hemocorr_pmd, device = 'cuda')

1
2
3
4
factorization started
completed


In [33]:
explained_var_spectrum_joao = get_pca_spectrum_pmd_obj(joao_hemocorr, device = 'cpu') ##Use cpu here, for some reason the gpu is acting weird here

1
2
3
4
factorization started


/data/home/app2139/masknmf-toolbox/masknmf/compression/decomposition.py:542: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  ut_u = torch.sparse.mm(u.T, u).to_dense()


completed


In [36]:
threshold = 0.95
explained_var = explained_var_spectrum_amol

# Find the first component that exceeds 95% explained variance
cutoff_idx = np.argmax(explained_var >= threshold)

plt.figure(figsize=(8, 6))
plt.plot(explained_var[:30], marker="o", label="Cumulative Explained Variance")
plt.axhline(y=threshold, color="red", linestyle="--", label="95% Threshold")
plt.axvline(x=cutoff_idx, color="gray", linestyle=":", label=f"Cutoff @ {cutoff_idx+1}")

# Formatting
plt.title("Hemocorrected PMD PCA spectrum", fontsize=14, weight="bold")
plt.xlabel("Number of Components", fontsize=12)
plt.ylabel("Cumulative Explained Variance", fontsize=12)
plt.xticks(range(0, 31, 2))
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()

plt.show()

In [37]:
threshold = 0.95
explained_var = explained_var_spectrum_joao

# Find the first component that exceeds 95% explained variance
cutoff_idx = np.argmax(explained_var >= threshold)

plt.figure(figsize=(8, 6))
plt.plot(explained_var[:30], marker="o", label="Cumulative Explained Variance")
plt.axhline(y=threshold, color="red", linestyle="--", label="95% Threshold")
plt.axvline(x=cutoff_idx, color="gray", linestyle=":", label=f"Cutoff @ {cutoff_idx+1}")

# Formatting
plt.title("Hemocorrected WFIELD PCA spectrum", fontsize=14, weight="bold")
plt.xlabel("Number of Components", fontsize=12)
plt.ylabel("Cumulative Explained Variance", fontsize=12)
plt.xticks(range(0, 31, 2))
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()

plt.show()